# Unidad III: modelos supervisados

## Objetivo

Entrenar y comparar modelos de regresión y clasificación con una partición de entrenamiento/prueba reproducible. Se usa el dataset procesado y se conserva la separación de variables categóricas y numéricas dentro de `scikit-learn`.

- **Regresión:** estimar `ProductRelated_Duration` en segundos. Es el objetivo numérico propuesto durante la exploración.
- **Clasificación:** predecir `Revenue` (si la sesión termina en compra).
- **Semilla:** 42 para poder reproducir las particiones.

Los registros duplicados y faltantes ya se revisaron en Unidad II. Los valores atípicos de duración se conservan: pueden representar sesiones largas reales; el bosque de árboles se incluye como alternativa al baseline para comparar su efecto.


In [2]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    mean_absolute_error, mean_squared_error, precision_score, recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ruta = Path('data/processed/online_shoppers_intention_clean.csv')
if not ruta.exists():
    ruta = Path('../data/processed/online_shoppers_intention_clean.csv')
if not ruta.exists():
    raise FileNotFoundError('No se encontró el dataset procesado.')

datos = pd.read_csv(ruta)
print(f'Dataset: {datos.shape[0]:,} sesiones x {datos.shape[1]} variables')
print('Proporción de compra:')
print(datos['Revenue'].value_counts(normalize=True).rename('proporción').round(3).to_string())

columnas_categoricas = [
    'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType',
    'VisitorType', 'Weekend'
]


Dataset: 12,205 sesiones x 18 variables
Proporción de compra:
Revenue
False    0.844
True     0.156


## 1. Regresión: duración en páginas de productos

`ProductRelated_Duration` se mide en segundos. Se comparan un predictor de referencia que siempre devuelve la mediana del entrenamiento y un `RandomForestRegressor`. Se excluye `Revenue` para no usar el resultado de compra como predictor de la duración de la sesión.

Las variables categóricas se codifican con one-hot; las numéricas se estandarizan dentro del pipeline. El preprocesamiento se ajusta solo con los datos de entrenamiento.


In [4]:
objetivo_regresion = 'ProductRelated_Duration'
X_reg = datos.drop(columns=[objetivo_regresion, 'Revenue'])
y_reg = datos[objetivo_regresion]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)
columnas_numericas_reg = [
    c for c in X_reg.select_dtypes(include='number').columns
    if c not in columnas_categoricas
]
preproceso_reg = ColumnTransformer([
    ('numericas', StandardScaler(), columnas_numericas_reg),
    ('categoricas', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas),
])

modelos_regresion = {
    'Baseline mediana': Pipeline([
        ('preparacion', preproceso_reg),
        ('modelo', DummyRegressor(strategy='median')),
    ]),
    'Random Forest': Pipeline([
        ('preparacion', preproceso_reg),
        ('modelo', RandomForestRegressor(
            n_estimators=120, min_samples_leaf=2, random_state=42, n_jobs=-1
        )),
    ]),
}

resultados_regresion = []
predicciones_regresion = {}
for nombre, modelo in modelos_regresion.items():
    modelo.fit(X_train_reg, y_train_reg)
    pred = modelo.predict(X_test_reg)
    predicciones_regresion[nombre] = pred
    resultados_regresion.append({
        'Modelo': nombre,
        'MAE_segundos': mean_absolute_error(y_test_reg, pred),
        'RMSE_segundos': mean_squared_error(y_test_reg, pred) ** 0.5,
        'R2': modelo.score(X_test_reg, y_test_reg),
    })

resultados_regresion = pd.DataFrame(resultados_regresion).sort_values('MAE_segundos')
print(resultados_regresion.round(3).to_string(index=False))
mejor_regresion = resultados_regresion.iloc[0]['Modelo']
print(f'Modelo con menor MAE: {mejor_regresion}')


          Modelo  MAE_segundos  RMSE_segundos     R2
   Random Forest       510.763       1141.955  0.715
Baseline mediana      1003.864       2225.911 -0.081
Modelo con menor MAE: Random Forest


### Lectura de las métricas de regresión

- **MAE** expresa el error absoluto promedio en segundos; valores menores son mejores.
- **RMSE** penaliza más los errores grandes, relevante porque las duraciones tienen una cola larga.
- **R²** compara la variación explicada con un baseline; puede ser negativo si el modelo predice peor que la media.

Compara el bosque con el baseline antes de concluir que aporta valor. El objetivo es duración de navegación, no ventas.


## 2. Clasificación: compra (`Revenue`)

`Revenue=True` es la clase positiva. La separación es estratificada para conservar la proporción de sesiones con compra en ambos subconjuntos. Se comparan un clasificador de referencia que siempre predice la clase mayoritaria, regresión logística y Random Forest.

Como hay menos sesiones con compra, la exactitud no basta. Se reportan precisión, recall, F1 y ROC AUC; el recall mide cuántas compras reales detecta el modelo.


In [7]:
X_clas = datos.drop(columns=['Revenue'])
y_clas = datos['Revenue'].astype(int)
X_train_clas, X_test_clas, y_train_clas, y_test_clas = train_test_split(
    X_clas, y_clas, test_size=0.20, random_state=42, stratify=y_clas
)
columnas_numericas_clas = [
    c for c in X_clas.select_dtypes(include='number').columns
    if c not in columnas_categoricas
]
preproceso_clas = ColumnTransformer([
    ('numericas', StandardScaler(), columnas_numericas_clas),
    ('categoricas', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas),
])

modelos_clasificacion = {
    'Baseline clase mayoritaria': Pipeline([
        ('preparacion', preproceso_clas),
        ('modelo', DummyClassifier(strategy='most_frequent')),
    ]),
    'Regresión logística': Pipeline([
        ('preparacion', preproceso_clas),
        ('modelo', LogisticRegression(
            max_iter=1500, class_weight='balanced', random_state=42
        )),
    ]),
    'Random Forest': Pipeline([
        ('preparacion', preproceso_clas),
        ('modelo', RandomForestClassifier(
            n_estimators=160, min_samples_leaf=3, class_weight='balanced_subsample',
            random_state=42, n_jobs=-1
        )),
    ]),
}

resultados_clasificacion = []
predicciones_clasificacion = {}
probabilidades_clasificacion = {}
for nombre, modelo in modelos_clasificacion.items():
    modelo.fit(X_train_clas, y_train_clas)
    pred = modelo.predict(X_test_clas)
    proba = modelo.predict_proba(X_test_clas)[:, 1]
    predicciones_clasificacion[nombre] = pred
    probabilidades_clasificacion[nombre] = proba
    resultados_clasificacion.append({
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_test_clas, pred),
        'Precision_compra': precision_score(y_test_clas, pred, zero_division=0),
        'Recall_compra': recall_score(y_test_clas, pred, zero_division=0),
        'F1_compra': f1_score(y_test_clas, pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test_clas, proba),
    })

resultados_clasificacion = pd.DataFrame(resultados_clasificacion).sort_values(
    ['F1_compra', 'Recall_compra'], ascending=False
)
print(resultados_clasificacion.round(3).to_string(index=False))
mejor_clasificacion = resultados_clasificacion.iloc[0]['Modelo']
print(f'Modelo seleccionado por F1 y recall: {mejor_clasificacion}')


                    Modelo  Accuracy  Precision_compra  Recall_compra  F1_compra  ROC_AUC
             Random Forest     0.896             0.650          0.730      0.688    0.929
       Regresión logística     0.846             0.505          0.796      0.618    0.910
Baseline clase mayoritaria     0.844             0.000          0.000      0.000    0.500
Modelo seleccionado por F1 y recall: Random Forest


In [8]:
pred_mejor = predicciones_clasificacion[mejor_clasificacion]
matriz_confusion = confusion_matrix(y_test_clas, pred_mejor, labels=[0, 1])
print('Matriz de confusión (filas = valor real; columnas = predicción):')
print(pd.DataFrame(
    matriz_confusion,
    index=['Real: no compra', 'Real: compra'],
    columns=['Predicho: no compra', 'Predicho: compra'],
).to_string())
print('Reporte por clase:')
print(classification_report(
    y_test_clas, pred_mejor,
    labels=[0, 1], target_names=['No compra', 'Compra'], zero_division=0
))


Matriz de confusión (filas = valor real; columnas = predicción):
                 Predicho: no compra  Predicho: compra
Real: no compra                 1909               150
Real: compra                     103               279
Reporte por clase:
              precision    recall  f1-score   support

   No compra       0.95      0.93      0.94      2059
      Compra       0.65      0.73      0.69       382

    accuracy                           0.90      2441
   macro avg       0.80      0.83      0.81      2441
weighted avg       0.90      0.90      0.90      2441



### Interpretación y límites

La comparación debe considerar la clase positiva `Revenue=True`. Un modelo puede tener alta exactitud y aun así no detectar compras; por eso se prioriza F1 junto con recall al escoger un candidato. La elección no implica que sea apto para producción: el corte aleatorio evalúa una sola partición y las variables describen sesiones, no personas identificadas.

La división de entrenamiento/prueba es reproducible con semilla 42, pero no sustituye validación cruzada ni análisis de costos de falsos positivos/falsos negativos.


In [10]:
# Guardar los pipelines ajustados y las métricas para reproducir/usar en el proyecto.
raiz = Path.cwd()
if raiz.name == 'notebooks':
    raiz = raiz.parent
carpeta_modelos = raiz / 'modelos'
carpeta_modelos.mkdir(parents=True, exist_ok=True)

joblib.dump(modelos_regresion[mejor_regresion], carpeta_modelos / 'regresion_product_duration.joblib', compress=3)
joblib.dump(modelos_clasificacion[mejor_clasificacion], carpeta_modelos / 'clasificacion_revenue.joblib', compress=3)

resultados_regresion.assign(Tarea='Regresión').to_csv(
    carpeta_modelos / 'metricas_regresion.csv', index=False
)
resultados_clasificacion.assign(Tarea='Clasificación').to_csv(
    carpeta_modelos / 'metricas_clasificacion.csv', index=False
)
print('Modelos y métricas guardados en:', carpeta_modelos.resolve())


Modelos y métricas guardados en: C:\Users\chave\REC_ECDB_2026_ChavezMolina_Byron\modelos


## Resultados obtenidos en esta partición

- **Regresión:** Random Forest obtuvo MAE de 510.76 segundos y R² de 0.715, frente a MAE de 1,003.86 segundos del baseline de mediana. El RMSE fue 1,141.95 segundos; los errores grandes siguen siendo relevantes por la cola larga de duración.
- **Clasificación:** Random Forest logró F1 de 0.688, recall de compra de 0.730 y ROC AUC de 0.929. Regresión logística tuvo recall más alto (0.796) y F1 de 0.618. El baseline alcanzó accuracy de 0.844 pero recall de compra de 0, así que la exactitud aislada no representa detección de compradores.

La selección del clasificador depende del objetivo: Random Forest equilibra mejor precisión/F1; si se prioriza detectar la mayor cantidad de compras, la regresión logística recupera más positivos en esta partición. Estos valores describen el conjunto de prueba separado con semilla 42 y no deben presentarse como rendimiento universal.



## Conclusión de Unidad III

El notebook define dos tareas supervisadas relacionadas con la pregunta del proyecto: estimar duración de navegación y clasificar si una sesión termina en compra. Se comparan contra baselines, se muestran métricas interpretables y se guardan los pipelines seleccionados y sus resultados en `modelos/`. Antes de defender el resultado, revisa las tablas impresas y explica qué errores son relevantes para cada tarea.
